In [4]:
# Load the fixed sampled dataset from disk
import json
from datasets import load_dataset
from pathlib import Path

data_path = Path("/home/a/arfaoui/rag_project/data/hotpotqa_full_sample.json")

dataset = load_dataset("json", data_files=str(data_path))["train"]

print("Loaded sample size:", len(dataset))
print("Columns:", dataset.column_names)



Loaded sample size: 7405
Columns: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


In [5]:
# We build a corpus from the HotpotQA contexts.
# Each document will be:
# - title + joined sentences
# - converted into a single text string

corpus = []
seen_titles = set()  # will be used to avoid duplicate articles
duplicate_count = 0
for example in dataset:
    context = example["context"]

    # IMPORTANT:
    # context is a dict with two parallel lists:
    # - context["title"]
    # - context["sentences"]
    # Observations from previous NoteBook

    for title, sentences in zip(context["title"], context["sentences"]):
         if title not in seen_titles:
            seen_titles.add(title)
            text = " ".join(sentences) # Join all sentences into one text block
            corpus.append({            # Store structured document
               "title": title,
               "text": text
        })
         else:
            duplicate_count += 1

# Print to check
print("Total documents in corpus no duplicates:", len(corpus))
print("\nExample document:\n")
print(corpus[0])
print("Number of duplicates:", duplicate_count)

Total documents in corpus no duplicates: 66581

Example document:

{'title': 'Ed Wood (film)', 'text': "Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast."}
Number of duplicates: 7119


In [6]:
# Save the corpus in data folder 
import json
from pathlib import Path

corpus_path = Path("/home/a/arfaoui/rag_project/data/corpus.json")

with open(corpus_path, "w", encoding="utf-8") as f:   # ensure_ascii=False -> important for any non-ASCII characters in HotpotQA
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print("Corpus saved to:", corpus_path)

Corpus saved to: /home/a/arfaoui/rag_project/data/corpus.json


### Corpus Construction Observations

- Each HotpotQA example contains multiple documents (≈10).
- Each document is defined by:
  - a title
  - a list of sentences
- We reconstruct documents by joining sentences into one text.
- Removing duplicates keeps the retrieval index cleaner and avoids storing the same article multiple times.
  
Result:
- The full validation HotpotQA questions contained 73700 document occurrences in total.
- After deduplication by title, the final corpus contains 66581 unique documents.
- 7119 duplicate article occurrences were removed.

  
Why this matters:
- This corpus is the basis for embedding + retrieval.
# Errors in this step break the entire RAG pipeline.